In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

all_sheets = pd.read_excel('/content/drive/MyDrive/Cadetx /ev_charging_dataset.xlsx', sheet_name=None)
sessions_df = all_sheets['sessions']

print(sessions_df.shape)

(294024, 15)


In [3]:
sessions_df['hour'] = sessions_df['start_timestamp'].dt.hour

hourly_pricing = sessions_df.groupby('hour').agg(
    session_count=('session_id', 'count'),
    avg_current_price=('price_per_kwh', 'mean')
).reset_index()

print(hourly_pricing)

    hour  session_count  avg_current_price
0      0           2754           0.579648
1      1           2677           0.578532
2      2           2719           0.579264
3      3           2633           0.579229
4      4           2705           0.580384
5      5           5360           0.577944
6      6           7985           0.578725
7      7          10648           0.578604
8      8          15967           0.579218
9      9          21370           0.579158
10    10          24214           0.579015
11    11          24048           0.578852
12    12          23945           0.580079
13    13          24123           0.579462
14    14          24120           0.579092
15    15          21251           0.579315
16    16          18750           0.637820
17    17          15992           0.637326
18    18          13480           0.635833
19    19          10628           0.636330
20    20           7959           0.636900
21    21           5353           0.578151
22    22   

In [4]:
hourly_revenue_current = sessions_df.groupby('hour')['total_cost'].sum().reset_index()
hourly_revenue_current.columns = ['hour', 'current_revenue']

print(hourly_revenue_current)

    hour  current_revenue
0      0         74539.92
1      1         72460.92
2      2         73800.06
3      3         71830.48
4      4         73769.08
5      5        144544.10
6      6        217232.60
7      7        288222.62
8      8        434377.18
9      9        581180.64
10    10        656239.85
11    11        653150.78
12    12        651031.16
13    13        655162.02
14    14        654884.69
15    15        581253.63
16    16        561397.38
17    17        477190.61
18    18        402675.13
19    19        315412.47
20    20        239021.86
21    21        146006.64
22    22         72766.94
23    23         72949.36


In [5]:
# Merge hourly session count and revenue together
pricing_sim = hourly_pricing.merge(hourly_revenue_current, on='hour')

# Define new premium hours (true peak: 10am-2pm) vs current premium hours (4pm-8pm)
new_premium_hours = [10, 11, 12, 13, 14]
old_premium_hours = [16, 17, 18, 19, 20]

# Apply +10% to new peak hours, reset old premium hours back to baseline
pricing_sim['simulated_revenue'] = pricing_sim['current_revenue']

pricing_sim.loc[pricing_sim['hour'].isin(new_premium_hours), 'simulated_revenue'] *= 1.10
pricing_sim.loc[pricing_sim['hour'].isin(old_premium_hours), 'simulated_revenue'] /= 1.10

print(pricing_sim[['hour', 'current_revenue', 'simulated_revenue']])

    hour  current_revenue  simulated_revenue
0      0         74539.92       74539.920000
1      1         72460.92       72460.920000
2      2         73800.06       73800.060000
3      3         71830.48       71830.480000
4      4         73769.08       73769.080000
5      5        144544.10      144544.100000
6      6        217232.60      217232.600000
7      7        288222.62      288222.620000
8      8        434377.18      434377.180000
9      9        581180.64      581180.640000
10    10        656239.85      721863.835000
11    11        653150.78      718465.858000
12    12        651031.16      716134.276000
13    13        655162.02      720678.222000
14    14        654884.69      720373.159000
15    15        581253.63      581253.630000
16    16        561397.38      510361.254545
17    17        477190.61      433809.645455
18    18        402675.13      366068.300000
19    19        315412.47      286738.609091
20    20        239021.86      217292.600000
21    21  

In [6]:
total_current = pricing_sim['current_revenue'].sum()
total_simulated = pricing_sim['simulated_revenue'].sum()

print(f"Current total revenue: ${total_current:,.2f}")
print(f"Simulated total revenue (repriced): ${total_simulated:,.2f}")
print(f"Net change: ${total_simulated - total_current:,.2f}")

Current total revenue: $8,171,100.12
Simulated total revenue (repriced): $8,316,719.93
Net change: $145,619.81


In [7]:
pricing_sim.to_csv('/content/drive/MyDrive/Cadetx /pricing_simulation.csv', index=False)
print("Saved!")

Saved!
